# End to End Deep Learning Project by using Simple RNN

In [55]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [56]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [57]:
from tensorflow.keras.preprocessing.text import one_hot

In [58]:
# Limit the vocabulary to the top 10,000 most frequent words
vocab_size = 10000

In [59]:

# Load the dataset split into training and testing sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

In [60]:
print(f"Training data shape is{X_train.shape}, Testing data shape is {X_test.shape}")

Training data shape is(25000,), Testing data shape is (25000,)


In [61]:
# Inspect a Sample Review and it's labels
sample_review = X_train[0]
sample_lable = y_train[0]
print(f"Sample review (as integer):{sample_review}")
print(f"Sample label:{sample_lable}")

Sample review (as integer):[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Sample label:1


In [62]:
from tensorflow.keras.preprocessing import sequence
max_len=300
X_train=sequence.pad_sequences(X_train, maxlen=max_len)
X_test=sequence.pad_sequences(X_test, maxlen=max_len)
X_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]], dtype=int32)

In [63]:
# Train simple RNN
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=128, input_shape=(max_len,)))
model.add(SimpleRNN(128,activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [64]:
model.compile(optimizer='adam',loss='binary_crossentropy',  metrics=['accuracy'])

In [65]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [66]:
earlystopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
earlystopping

In [67]:
# Train the model with early stopping
history = model.fit(
    X_train,y_train,epochs=10,batch_size = 32,
    validation_split = 0.2,
    callbacks = [earlystopping]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 28ms/step - accuracy: 0.5953 - loss: 3.5069 - val_accuracy: 0.6004 - val_loss: 0.6436
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6955 - loss: 9051370.0000 - val_accuracy: 0.7660 - val_loss: 0.4797
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.8213 - loss: 0.5206 - val_accuracy: 0.8188 - val_loss: 0.4142
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.8951 - loss: 0.2689 - val_accuracy: 0.8340 - val_loss: 0.4474
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.9285 - loss: 0.1993 - val_accuracy: 0.8290 - val_loss: 0.4087
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.9431 - loss: 0.1577 - val_accuracy: 0.8338 - val_loss: 0.4516
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.9582 - loss: 0.1187 - val_accuracy: 0.8388 - val_loss: 0.5140


In [68]:
# Save Model file
model.save("simple_rnn_imdb.h5")

# Model Predictions

In [70]:
# Load the IMDB dataset word index
from tensorflow.keras.datasets import imdb
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [73]:
# Load the pre-trained model with ReLU activation
from tensorflow.keras.models import load_model
model = load_model('/content/simple_rnn_imdb.h5')
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 300, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [74]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])
# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences ([encoded_review], maxlen=500)
    return padded_review

In [75]:
### Prediction function
def predict_sentiment (review):
    preprocessed_input=preprocess_text(review)
    prediction=model.predict(preprocessed_input)
    sentiment = 'Positive' if prediction[0][0] > 0.6 else 'Negative'
    return sentiment, prediction [0][0]

In [76]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."
sentiment, score= predict_sentiment(example_review)
print (f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print (f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 469ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Prediction Score: 0.7795589566230774


In [79]:
# Example2 review for prediction
example_review_1 = "This movie was very bad and so boring characters"
sentiment, score= predict_sentiment(example_review_1)
print (f'Review: {example_review_1}')
print(f'Sentiment: {sentiment}')
print (f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Review: This movie was very bad and so boring characters
Sentiment: Negative
Prediction Score: 0.09780100733041763
